### 1. 📥​ Imports

In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
from pathlib import Path
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

from tcibench.preparation import (Dataset, DatasetMetadata, load_dataset, load_metadata)

from tcibench.generation import (
    generate_with_synthcity,
    balance_data
)

/home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### 2. ⚙️​ Configs

In [3]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "tcibench" / "preparation" / "metadata.json").is_file()),
    None,
)

In [4]:
# Reproducibility
SPLIT_SEED = 42
SAMPLING_SEED = 42
GENERATION_SEED = 42

# Generation protocol
EXAMPLES_PER_CLASS = 10
RECORDS_PER_BATCH = 10

📂​ Datasets

In [5]:
DATASETS = {
    "diabetes": "Diabetes",
    "sick": "Thyroid Sick",
    "predictive_maintenance": "Predictive Maintenance",
}

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / f"split_seed_{SPLIT_SEED}"

RAW_PATHS = {
    "diabetes": RAW_ROOT / "diabetes.csv",
    "sick": RAW_ROOT / "sick.arff",
    "predictive_maintenance": RAW_ROOT / "predictive_maintenance.csv",
}

🤖 Models

In [6]:
GENERATIVE_MODELS = ["ddpm", "ctgan", "tvae"]
CLASSICAL_METHODS = ["ros", "rus", "smotenc"]

### 3. 📂​ Dataset

In [7]:
def inspect_dataset(
    data: pd.DataFrame,
    metadata: DatasetMetadata,
    cleaned: pd.DataFrame,
    cleaning_stats: dict,
):
    """Inspect a dataset before running the experiment."""

    print(f"Dataset: {metadata.display_name}")
    print(f"Target: {metadata.target_column}")

    print("\nRaw data")
    print(f"Shape: {data.shape}")
    display(data.head())

    print("\nCleaning")
    print(f"Shape after cleaning: {cleaned.shape}")
    print(f"Removed columns: {cleaning_stats['dropped_columns']}")
    print(f"Removed rows with missing values: {cleaning_stats['removed_missing_rows']}")
    print(f"Removed duplicate rows: {cleaning_stats['removed_duplicate_rows']}")

    print("\nCleaned data")
    display(cleaned.head())

    print("\nTarget distribution")
    target = metadata.target_column
    display(pd.DataFrame({
        "count": cleaned[target].value_counts(),
        "proportion": cleaned[target].value_counts(normalize=True).mul(100).round(2),
    }))

    target_counts = cleaned[target].value_counts()
    print(f"Imbalance ratio: {target_counts.max() / target_counts.min():.2f}")

In [8]:
DATASET_ID = "predictive_maintenance"

metadata = load_metadata(DATASET_ID)
data = load_dataset(RAW_PATHS[DATASET_ID])

dataset = Dataset(data, metadata)

cleaned, cleaning_report = dataset.preprocessing.clean(data)
folds = dataset.preprocessing.split(cleaned)

inspect_dataset(data, metadata, cleaned, cleaning_report)

train, test, audit = folds[0]

counts = train[metadata.target_column].value_counts()
majority_class = counts.idxmax()
minority_class = counts.idxmin()

majority_count = int(counts[majority_class])
minority_count = int(counts[minority_class])
records_to_generate = majority_count - minority_count
imbalance_ratio = majority_count / minority_count

print(f"Dataset: {metadata.display_name}")
print(f"Cleaned: {cleaned.shape}")
print(f"Fold 1 — Train: {train.shape} | Test: {test.shape}")
print(f"Majority: {majority_class} ({majority_count})")
print(f"Minority: {minority_class} ({minority_count})")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"Records to generate: {records_to_generate}")

Dataset: AI4I 2020 Predictive Maintenance
Target: Target

Raw data
Shape: (10000, 10)


,UDI,Product ID,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure



Cleaning
Shape after cleaning: (10000, 7)
Removed columns: ['UDI', 'Product ID', 'Failure Type']
Removed rows with missing values: 0
Removed duplicate rows: 0

Cleaned data


,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Target
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0



Target distribution


,count,proportion
Target,,
0,9661,96.61
1,339,3.39


Imbalance ratio: 28.50
Dataset: AI4I 2020 Predictive Maintenance
Cleaned: (10000, 7)
Fold 1 — Train: (8000, 7) | Test: (2000, 7)
Majority: 0 (7728)
Minority: 1 (272)
Imbalance ratio: 28.41
Records to generate: 7456


### 4. 🤖 Models generation

In [9]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import torch
from synthcity.utils.constants import DEVICE

print("PyTorch:", torch.cuda.is_available())
print("Synthcity:", DEVICE)

True
NVIDIA GeForce RTX 4060 Ti
PyTorch: True
Synthcity: cuda


In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "generation"
report_path = (
    OUTPUT_ROOT / metadata.dataset_id / "baselines" / "generative_stats.json"
)
report_path.parent.mkdir(parents=True, exist_ok=True)

all_stats = []

for fold_id, (train, test, audit) in enumerate(folds):
    counts = train[metadata.target_column].value_counts()
    minority_class = counts.idxmin()
    majority_class = counts.idxmax()

    n_samples = int(
        counts[majority_class] - counts[minority_class]
    )

    for model_name in GENERATIVE_MODELS:
        synthetic, stats = generate_with_synthcity(
            train,
            test=test,
            metadata=metadata,
            model_name=model_name,
            n_samples=n_samples,
            random_seed=GENERATION_SEED,
            target_column=metadata.target_column,
            minority_class=minority_class,
            fold_id=fold_id,
            output_root=OUTPUT_ROOT,
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep completed runs if a later model fails.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Model: {model_name} | "
            f"Minority: {minority_class} | "
            f"Generated: {len(synthetic)}"
        )

        display(stats)

[2026-10-14T20:12:54.234028-0300][714443][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-10-14T20:12:54.237644-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 1000/1000 [01:42<00:00,  9.75it/s, loss=1.29]


ddpm | fold 0 | attempt 1/10 | generated 8202 | wrong class 0 | duplicates 5 | accumulated 1.10  classes {1: 8202} | copies of train 0
Fold 0 | Model: ddpm | Minority: 1 | Generated: 7456


,fold,model,minority_class,random_seed,generation_seed,needed,requested,generated,valid,wrong_class,duplicates,excess,attempts,copies_train,copies_test
0,0,ddpm,1,42,42,7456,8202,8202,8197,0,5,741,1,0,0


[2026-10-14T20:14:57.843642-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
 22%|██▏       | 449/2000 [03:52<13:21,  1.93it/s] 

ctgan | fold 0 | attempt 1/10 | generated 8202 | wrong class 525 | duplicates 0 | accumulated 1.03  classes {1: 7677, 0: 525} | copies of train 0
Fold 0 | Model: ctgan | Minority: 1 | Generated: 7456


,fold,model,minority_class,random_seed,generation_seed,needed,requested,generated,valid,wrong_class,duplicates,excess,attempts,copies_train,copies_test
0,0,ctgan,1,42,42,7456,8202,8202,7677,525,0,221,1,0,0


[2026-10-14T20:18:53.539474-0300][714443][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
 76%|███████▋  | 763/1000 [03:29<01:06,  3.55it/s]

In [ ]:
for fold_id, (train, test, audit) in enumerate(folds):
    for method in CLASSICAL_METHODS:
        actual_method = method
        if method == "smotenc" and not any(
            column in train.columns
            for column in metadata.categorical_columns
        ):
            actual_method = "smote"

        balanced_train, stats = balance_data(
            train,
            test=test,
            metadata=metadata,
            method=method,
            fold_id=fold_id,
            random_seed=GENERATION_SEED,
            output_root=PROJECT_ROOT / "artifacts" / "generation"
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep one report with generative and classical baseline statistics.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Method: {actual_method} | "
            f"Rows saved: {len(balanced_train)}"
        )

FileExistsError: Output already exists: /home/labian-3/Documentos/Fernanda/tcibench/artifacts/generation/sick/baselines/ros/fold_0.csv